In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Config
Toàn bộ tham số dự án — chỉ chỉnh tại đây.

In [ ]:
import os, sys, random, unicodedata, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import AdamW
from tensorboardX import SummaryWriter
from tqdm.notebook import tqdm, trange
from sklearn.metrics import f1_score, classification_report, confusion_matrix, f1_score as sk_f1
from collections import Counter
from underthesea import word_tokenize


try:
    import gensim
except ImportError:
    print("Installing gensim...")
    !pip install gensim -q
    import gensim


BASE_DIR = '/content/drive/MyDrive/bilstm'

W2V_PATH = f'{BASE_DIR}/phow2v_300d.txt'

train1     = f'{BASE_DIR}/train1.csv'
train2     = f'{BASE_DIR}/train2.csv'
train3     = f'{BASE_DIR}/train3.csv'
train4     = f'{BASE_DIR}/train4.csv'
train5     = f'{BASE_DIR}/train5.csv'
train6     = f'{BASE_DIR}/train6.csv'
validation = f'{BASE_DIR}/validation.csv'
test       = f'{BASE_DIR}/test.csv'


OUTPUT_DIR = f'{BASE_DIR}/a'
os.makedirs(OUTPUT_DIR, exist_ok=True)


EMBED_DIM = 100
HIDDEN_DIM = 256
CNN_FILTERS = 256
KERNEL_SIZE = 3

SEED = 42
MAX_EPOCHS = 35
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5
MAX_SEQ_LENGTH = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng device: {device}")
print(f"Thư mục lưu trữ: {OUTPUT_DIR}")

## 2. Kiểm tra GPU

## 2. Kiểm tra GPU

In [ ]:
import torch


ASPECT_CATEGORIES = [
    'ky_nang_giang_day', 'hanh_vi', 'de_xuat', 'bai_tap',
    'chuong_trinh_hoc', 'kien_thuc', 'kinh_nghiem',
    'cung_cap_tai_lieu', 'thiet_bi_day_hoc', 'cham_diem', 'noi_chung'
]
POLARITIES = ['negative', 'neutral', 'positive']

ASPECT2ID = {cat: i for i, cat in enumerate(ASPECT_CATEGORIES)}
POL2ID    = {pol: i for i, pol in enumerate(POLARITIES)}
ID2POL    = {i: pol for pol, i in POL2ID.items()}

NUM_ASPECTS = len(ASPECT_CATEGORIES)
NUM_POLARITIES = len(POLARITIES)
IGNORE_INDEX = -100


ALPHA = 1.0
BETA  = 2.0

# Cấu hình log
LOG_STEPS = 50
SAVE_STEPS = 500

print('PyTorch version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpu  = torch.cuda.device_count()

print(f'\nSố lượng khía cạnh (Aspects): {NUM_ASPECTS}')
print(f'Số lượng cảm xúc (Polarities): {NUM_POLARITIES}')

## 3. Data — Convert CSV → Category CSV

## 3.1. Word Segmentation

In [ ]:
def preprocess_text(text):
    """
    Tiền xử lý cho PhoBERT:
    1. Chuẩn hóa Unicode NFC
    2. Trim whitespace thừa, chuẩn hóa dấu câu đầu câu
    3. Word segmentation → underscore (giảng viên → giảng_viên)
    """

    text = unicodedata.normalize('NFC', str(text).strip())

    # 2. Bỏ dấu chấm/khoảng trắng thừa ở đầu câu (vd: ". Thầy...")
    text = re.sub(r'^[\s\.\-]+', '', text)

    # 3. Word segmentation
    text = word_tokenize(text, format='text')   # "giảng viên" → "giảng_viên"

    return text


# Test thử
samples_test = [
    'Giảng viên rất nhiệt tình và tài năng.',
    'Phòng học quá nóng, không có điều hòa.',
    '. Thầy/cô luôn tôn trọng sinh viên.',
]
print('Word segmentation examples:')
for s in samples_test:
    print(f'  Before: {s}')
    print(f'  After : {preprocess_text(s)}')
    print()


In [ ]:
df_train1 = pd.read_csv(train1)
df_train2 = pd.read_csv(train2)
df_train3 = pd.read_csv(train3)
df_train4 = pd.read_csv(train4)
df_train5 = pd.read_csv(train5)
df_train6 = pd.read_csv(train6)

df_validation = pd.read_csv(validation)
df_test = pd.read_csv(test)

print(df_train1.info())

In [ ]:
df = pd.concat([df_train1, df_train2, df_train3, df_train4, df_train5, df_train6], ignore_index=True)

In [ ]:
df.head(5)

In [ ]:
print(df.value_counts("aspect"))

In [ ]:
print(df.value_counts("sentiment"))

In [ ]:
min_len = df['text'].str.len().min()
max_len = df['text'].str.len().max()

print("Min length:", min_len)
print("Max length:", max_len)

In [ ]:
df['text'].str.len().describe()

In [ ]:
import matplotlib.pyplot as plt

lengths = df['text'].str.len()

plt.figure()
plt.hist(lengths, bins=50)
plt.title("Distribution of Text Length")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.show()

In [ ]:
token_lens = []
for text in df['text']:
    processed = preprocess_text(text)   # phải preprocess trước như lúc train
    tokens = tokenizer.encode(processed, add_special_tokens=True)
    token_lens.append(len(tokens))

pd.Series(token_lens).describe()


In [ ]:
import numpy as np

print("90%:", np.percentile(token_lens, 90))
print("95%:", np.percentile(token_lens, 95))
print("99%:", np.percentile(token_lens, 99))
print("max:", max(token_lens))

In [ ]:
def load_acsa_data(df_file):
    """
    Group theo id → mỗi sample:
      - text      : str (đã preprocess)
      - aspect    : (NUM_ASPECTS,) float multi-hot
      - sentiment : (NUM_ASPECTS,) int, IGNORE_INDEX nếu absent
    """
    df = df_file
    samples = []
    skipped = 0

    for sent_id, group in df.groupby('id'):
        text = preprocess_text(group.iloc[0]['text'])

        aspect_labels    = np.zeros(NUM_ASPECTS, dtype=np.float32)
        sentiment_labels = np.full(NUM_ASPECTS, IGNORE_INDEX, dtype=np.int64)

        for _, row in group.iterrows():
            asp_str = str(row['aspect']).strip()
            sen_str = str(row['sentiment']).strip()

            if asp_str not in ASPECT2ID:
                skipped += 1
                continue
            if sen_str not in POL2ID:
                skipped += 1
                continue

            asp_idx = ASPECT2ID[asp_str]
            sen_idx = POL2ID[sen_str]

            aspect_labels[asp_idx] = 1.0
            if sentiment_labels[asp_idx] == IGNORE_INDEX:
                sentiment_labels[asp_idx] = sen_idx

        samples.append({
            'id':       sent_id,
            'text':     text,
            'aspect':   aspect_labels,
            'sentiment': sentiment_labels,
        })

    if skipped:
        print(f'  ⚠️  Skipped {skipped} rows with unknown aspect/sentiment labels')
    return samples


def print_stats(samples, split_name):
    print(f'=====================================================')
    print(f'  {split_name}: {len(samples)} sentences')
    print(f'=====================================================')
    aspect_counts = np.zeros(NUM_ASPECTS)
    pol_counts    = Counter()
    for s in samples:
        aspect_counts += s['aspect']
        for i, v in enumerate(s['sentiment']):
            if v != IGNORE_INDEX:
                pol_counts[ID2POL[v]] += 1
    print('  Aspect distribution:')
    max_cnt = aspect_counts.max()
    for i, cat in enumerate(ASPECT_CATEGORIES):
        cnt = int(aspect_counts[i])
        bar = '█' * int(30 * cnt / (max_cnt or 1))
        print(f'    {i:2d}. {cat:<25s} {cnt:4d}  {bar}')
    print('\n  Polarity distribution:')
    total_pol = sum(pol_counts.values())
    for pol in POLARITIES:
        cnt = pol_counts[pol]
        bar = '█' * int(30 * cnt / (total_pol or 1))
        print(f'    {pol:<12s} {cnt:4d} ({100*cnt/(total_pol or 1):.1f}%)  {bar}')


load_train1 = load_acsa_data(df_train1)
load_train2 = load_acsa_data(df_train2)
load_train3 = load_acsa_data(df_train3)
load_train4 = load_acsa_data(df_train4)
load_train5 = load_acsa_data(df_train5)
load_train6 = load_acsa_data(df_train6)

train_s = load_train1 + load_train2 + load_train3 + load_train4 + load_train5 + load_train6
dev_s   = load_acsa_data(df_validation)
test_s  = load_acsa_data(df_test)


for split in [train_s, dev_s, test_s]:
    for item in split:
        item.pop('id', None)

print_stats(train_s, 'Train')
print_stats(dev_s,   'Dev')
print_stats(test_s,  'Test')


# ── Verify IGNORE_INDEX ───────────────────────────────────────────────────
def verify_labels(samples, split_name):
    errors = 0
    for s in samples:
        for i in range(NUM_ASPECTS):
            asp = s['aspect'][i]
            sen = s['sentiment'][i]
            if asp == 0 and sen != IGNORE_INDEX:
                errors += 1
            if asp == 1 and sen not in [0, 1, 2]:
                errors += 1
    status = '✓' if errors == 0 else f'⚠️  {errors} errors'
    print(f'  {split_name} label check: {status}')

verify_labels(train_s, 'Train')
verify_labels(dev_s,   'Dev')
verify_labels(test_s,  'Test')


In [ ]:
print("Loading Pre-trained Word2Vec...")
w2v_model = gensim.models.KeyedVectors.load_word2vec_format(W2V_PATH, binary=False)


PAD_TOKEN, UNK_TOKEN = '<pad>', '<unk>'
word2id = {PAD_TOKEN: 0, UNK_TOKEN: 1}
vocab_words = [PAD_TOKEN, UNK_TOKEN]

for s in train_s:
    for w in s['text'].split():
        if w not in word2id:
            word2id[w] = len(word2id)
            vocab_words.append(w)

VOCAB_SIZE = len(word2id)
print(f"Vocab Size: {VOCAB_SIZE}")


embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM))
unk_count = 0

for i, word in enumerate(vocab_words):
    if i == 0: continue # Skip padding
    if word in w2v_model:
        embedding_matrix[i] = w2v_model[word]
    else:

        embedding_matrix[i] = np.random.normal(scale=0.1, size=(EMBED_DIM,))
        unk_count += 1

print(f"UNK words (initialized randomly): {unk_count}")
embedding_tensor = torch.tensor(embedding_matrix, dtype=torch.float32)

## 4. Dataset & DataLoader — từ Category CSV

In [ ]:
class ACSADataset(Dataset):
    def __init__(self, samples, word2id, max_len):
        self.samples = samples
        self.word2id = word2id
        self.max_len = max_len

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        words = s['text'].split()


        seq = [self.word2id.get(w, self.word2id['<unk>']) for w in words]

        # Cắt ngắn hoặc Đệm (Padding)
        if len(seq) < self.max_len:
            # Tạo dummy attention_mask
            mask = [1] * len(seq) + [0] * (self.max_len - len(seq))
            seq = seq + [self.word2id['<pad>']] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(seq, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.float),
            'aspect_labels': torch.tensor(s['aspect'], dtype=torch.float),
            'sentiment_labels': torch.tensor(s['sentiment'], dtype=torch.long),
        }

# Khởi tạo lại Dataloader
train_dataset = ACSADataset(train_s, word2id, MAX_SEQ_LENGTH)
dev_dataset   = ACSADataset(dev_s,   word2id, MAX_SEQ_LENGTH)
test_dataset  = ACSADataset(test_s,  word2id, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=TRAIN_BATCH_SIZE)
dev_loader   = DataLoader(dev_dataset,   sampler=SequentialSampler(dev_dataset), batch_size=EVAL_BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  sampler=SequentialSampler(test_dataset), batch_size=EVAL_BATCH_SIZE)

## 5. Model — BertMultiTaskABSA

In [ ]:

all_asp    = np.array([s['aspect'] for s in train_s])
pos_counts = all_asp.sum(axis=0)
neg_counts = len(all_asp) - pos_counts
pw_raw     = neg_counts / (pos_counts + 1e-8)
pw_clipped = np.clip(pw_raw, 1.0, 20.0)          # clip max 20
pw         = torch.tensor(pw_clipped, dtype=torch.float)
print('pos_weight (clipped):', pw.numpy().round(1))


def mean_pooling(last_hidden_state, attention_mask):
    """Mean pooling theo attention mask — ổn định hơn pooler_output."""
    mask   = attention_mask.unsqueeze(-1).float()          # (B, L, 1)
    summed = (last_hidden_state * mask).sum(dim=1)         # (B, H)
    counts = mask.sum(dim=1).clamp(min=1e-9)               # (B, 1)
    return summed / counts                                  # (B, H)


class BiLSTMCNNMultiTaskABSA(nn.Module):
    def __init__(self, embedding_tensor, hidden_dim, cnn_filters, kernel_size,
                 num_aspects, num_polarities, pos_weight, dropout=0.5):
        super().__init__()
        self.num_aspects = num_aspects
        self.num_polarities = num_polarities
        self.register_buffer('pos_weight', pos_weight)


        self.embedding = nn.Embedding.from_pretrained(embedding_tensor, freeze=True, padding_idx=0)
        embed_dim = embedding_tensor.shape[1]


        self.embed_drop = nn.Dropout2d(dropout/2)


        self.bilstm = nn.LSTM(embed_dim, hidden_dim, num_layers=1,
                              bidirectional=True, batch_first=True)


        self.conv1d = nn.Conv1d(in_channels=hidden_dim * 2, out_channels=cnn_filters,
                                kernel_size=kernel_size, padding=kernel_size//2)

        self.dropout = nn.Dropout(dropout)


        self.aspect_head = nn.Linear(cnn_filters, num_aspects)
        self.sentiment_head = nn.Linear(cnn_filters, num_aspects * num_polarities)

    def forward(self, input_ids, attention_mask, aspect_labels=None, sentiment_labels=None):

        x = self.embedding(input_ids) # (B, L, E)

        # Spatial Dropout
        x = x.unsqueeze(2)
        x = self.embed_drop(x)
        x = x.squeeze(2)

        # BiLSTM
        lstm_out, _ = self.bilstm(x)  # (B, L, H*2)

        # CNN
        lstm_out = lstm_out.transpose(1, 2) # (B, H*2, L)
        cnn_out = F.relu(self.conv1d(lstm_out)) # (B, Filters, L)

        # Global Max Pooling
        pooled = F.max_pool1d(cnn_out, cnn_out.size(2)).squeeze(2) # (B, Filters)
        pooled = self.dropout(pooled)

        # Heads
        asp_logits = self.aspect_head(pooled)
        sent_logits = self.sentiment_head(pooled).view(-1, self.num_aspects, self.num_polarities)

        loss = None
        if aspect_labels is not None and sentiment_labels is not None:
            loss_aspect = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)(asp_logits, aspect_labels)
            loss_sent = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)(
                sent_logits.view(-1, self.num_polarities), sentiment_labels.view(-1))
            loss = ALPHA * loss_aspect + BETA * loss_sent

        return loss, asp_logits, sent_logits

# Khởi tạo mô hình
model = BiLSTMCNNMultiTaskABSA(
    embedding_tensor=embedding_tensor,
    hidden_dim=HIDDEN_DIM,
    cnn_filters=CNN_FILTERS,
    kernel_size=KERNEL_SIZE,
    num_aspects=NUM_ASPECTS,
    num_polarities=NUM_POLARITIES,
    pos_weight=pw
)
model.to(device)
print(f'✓ Model loaded | Params cần train: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 6. Training

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    total_loss   = 0.0
    all_asp_gold, all_asp_pred = [], []
    all_sen_gold, all_sen_pred = [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating', leave=False):
            input_ids        = batch['input_ids'].to(device)
            attention_mask   = batch['attention_mask'].to(device)
            aspect_labels    = batch['aspect_labels'].to(device)
            sentiment_labels = batch['sentiment_labels'].to(device)

            loss, asp_logits, sent_logits = model(
                input_ids, attention_mask,
                aspect_labels, sentiment_labels)
            if n_gpu > 1: loss = loss.mean()
            total_loss += loss.item()

            # Aspect: threshold 0.5
            asp_pred = (torch.sigmoid(asp_logits) > 0.5).cpu().numpy().astype(int)
            asp_gold = aspect_labels.cpu().numpy().astype(int)
            all_asp_gold.append(asp_gold)
            all_asp_pred.append(asp_pred)

            # Sentiment: argmax, chỉ tính ở vị trí present
            sent_pred = torch.argmax(sent_logits, dim=-1).cpu().numpy()  # (B, A)
            sent_gold = sentiment_labels.cpu().numpy()                    # (B, A)
            mask      = sent_gold != IGNORE_INDEX
            all_sen_gold.extend(sent_gold[mask].tolist())
            all_sen_pred.extend(sent_pred[mask].tolist())

    all_asp_gold = np.vstack(all_asp_gold)
    all_asp_pred = np.vstack(all_asp_pred)

    aspect_f1 = f1_score(all_asp_gold, all_asp_pred, average='micro', zero_division=0)
    sent_f1   = f1_score(all_sen_gold, all_sen_pred, average='macro', zero_division=0)

    return {
        'loss':      total_loss / len(loader),
        'aspect_f1': aspect_f1,
        'sent_f1':   sent_f1,
        'asp_gold':  all_asp_gold,
        'asp_pred':  all_asp_pred,
        'sen_gold':  all_sen_gold,
        'sen_pred':  all_sen_pred,
    }


In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)


optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

tb_writer   = SummaryWriter(log_dir=OUTPUT_DIR)
global_step = 0
history     = {'epoch': [], 'train_loss': [], 'val_loss': [], 'val_aspect_f1': [], 'val_sent_f1': [], 'val_combined_f1': []}

# Early stopping
best_val_f1    = 0.0
patience_count = 0
best_ckpt_path = os.path.join(OUTPUT_DIR, 'best_model.pt')

for epoch in trange(MAX_EPOCHS, desc='Epoch'):
    model.train()
    epoch_loss, epoch_steps = 0.0, 0

    for batch in tqdm(train_loader, desc=f'Train E{epoch+1}', leave=False):
        input_ids        = batch['input_ids'].to(device)
        attention_mask   = batch['attention_mask'].to(device)
        aspect_labels    = batch['aspect_labels'].to(device)
        sentiment_labels = batch['sentiment_labels'].to(device)

        loss, _, _ = model(input_ids, attention_mask, aspect_labels, sentiment_labels)
        if n_gpu > 1: loss = loss.mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); model.zero_grad()

        epoch_loss  += loss.item()
        epoch_steps += 1
        global_step += 1

        if global_step % LOG_STEPS == 0:
            tb_writer.add_scalar('train_loss', loss.item(), global_step)
            tb_writer.add_scalar('lr', scheduler.get_last_lr()[0], global_step)

        if global_step % SAVE_STEPS == 0:
            ckpt = os.path.join(OUTPUT_DIR, f'checkpoint-{global_step}')
            os.makedirs(ckpt, exist_ok=True)
            m = model.module if hasattr(model, 'module') else model
            torch.save(m.state_dict(), os.path.join(ckpt, 'model.pt'))

    # ── Eval cuối epoch ──────────────────────────────────────────────────
    val_metrics = evaluate_model(model, dev_loader)
    avg_train   = epoch_loss / epoch_steps

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(avg_train)
    history['val_loss'].append(val_metrics['loss'])
    history['val_aspect_f1'].append(val_metrics['aspect_f1'])
    history['val_sent_f1'].append(val_metrics['sent_f1'])

    combined_f1 = (val_metrics['aspect_f1'] * val_metrics['sent_f1']) ** 0.5
    history['val_combined_f1'].append(combined_f1)


    scheduler.step(combined_f1)

    tb_writer.add_scalar('val_loss',      val_metrics['loss'],      global_step)
    tb_writer.add_scalar('val_aspect_f1', val_metrics['aspect_f1'], global_step)
    tb_writer.add_scalar('val_sent_f1',   val_metrics['sent_f1'],   global_step)
    tb_writer.add_scalar('val_combined_f1', combined_f1,              global_step)

    print(f"[Epoch {epoch+1}] train_loss={avg_train:.4f} | "
          f"val_loss={val_metrics['loss']:.4f} | "
          f"aspect_f1={val_metrics['aspect_f1']:.4f} | "
          f"sent_f1={val_metrics['sent_f1']:.4f} | "
          f"combined_f1={combined_f1:.4f}")

    # ── Lưu best model ───────
    if combined_f1 > best_val_f1:
        best_val_f1 = combined_f1
        patience_count = 0
        m = model.module if hasattr(model, 'module') else model
        torch.save(m.state_dict(), best_ckpt_path)
        print(f'  ✓ Best model saved (combined_f1={best_val_f1:.4f}  |  asp={val_metrics["aspect_f1"]:.4f}  sent={val_metrics["sent_f1"]:.4f})')
    else:
        patience_count += 1
        print(f'  No improvement ({patience_count}/{PATIENCE})')
        if patience_count >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

tb_writer.close()
# Lưu model cuối
m = model.module if hasattr(model, 'module') else model
torch.save(m.state_dict(), os.path.join(OUTPUT_DIR, 'model_final.pt'))
print(f'\n✓ Training done. Best val_combined_f1={best_val_f1:.4f}')
print(f'  Best model: {best_ckpt_path}')
print(f'  Final model: {OUTPUT_DIR}/model_final.pt')


## 7. Evaluation

In [ ]:
print('=== Dev ===')
dev_metrics  = evaluate_model(model, dev_loader)
print(f"aspect_f1={dev_metrics['aspect_f1']:.4f} | sent_f1={dev_metrics['sent_f1']:.4f}")

print('\n=== Test ===')
test_metrics = evaluate_model(model, test_loader)
print(f"aspect_f1={test_metrics['aspect_f1']:.4f} | sent_f1={test_metrics['sent_f1']:.4f}")

# Per-category sentiment report
print('\nPer-class Sentiment (Test):')
print(classification_report(
    test_metrics['sen_gold'], test_metrics['sen_pred'],
    target_names=POLARITIES, zero_division=0))


## 8. Visualization

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np

# ── 1. Learning curves (overfit / underfit) ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Learning Curves', fontsize=14, fontweight='bold')

epochs = history['epoch']
ax1.plot(epochs, history['train_loss'], 'o-',  color='#3498db', label='Train Loss', lw=2)
ax1.plot(epochs, history['val_loss'],   's--', color='#e74c3c', label='Val Loss',   lw=2)
gap = history['val_loss'][-1] - history['train_loss'][-1]
ax1.annotate(f'gap={gap:+.3f}',
             xy=(epochs[-1], history['val_loss'][-1]),
             xytext=(-40, 12), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='gray'), fontsize=9, color='#e74c3c')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss per Epoch')
ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.5); ax1.set_axisbelow(True)

best_epoch = epochs[history['val_combined_f1'].index(max(history['val_combined_f1']))]
best_f1    = max(history['val_combined_f1'])
ax2.plot(epochs, history['val_aspect_f1'],   's-', color='#2ecc71', lw=2, label='Val Aspect F1')
ax2.plot(epochs, history['val_sent_f1'],     'o-', color='#e67e22', lw=2, label='Val Sent F1')
ax2.plot(epochs, history['val_combined_f1'], 'D-', color='#9b59b6', lw=2, label='Val Combined F1')
ax2.axvline(best_epoch, color='gray', linestyle=':', lw=1.5)
ax2.scatter([best_epoch], [best_f1], color='#f39c12', zorder=5, s=80,
            label=f'Best Combined={best_f1:.3f} @ ep{best_epoch}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1 Score'); ax2.set_title('Val F1 Metrics per Epoch (Combined = Geometric Mean)')
ax2.legend(fontsize=8); ax2.grid(True, linestyle='--', alpha=0.5); ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'learning_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

if gap > 0.3:
    print(f"⚠️  Overfit: gap={gap:.3f} — thử tăng dropout hoặc giảm MAX_EPOCHS.")
elif best_f1 < 0.5:
    print(f"⚠️  Underfit: best_f1={best_f1:.3f} — thử tăng MAX_EPOCHS hoặc giảm LR.")
else:
    print(f"✓  Fit tốt: gap={gap:.3f}, best aspect_f1={best_f1:.3f} @ epoch {best_epoch}.")

# ── 2. Dev vs Test metrics bar chart ─────────────────────────────────────
metric_keys = ['aspect_f1', 'sent_f1', 'loss']
summary = pd.DataFrame([
    {'Split': 'Dev',  **{k: round(dev_metrics[k],  4) for k in metric_keys}},
    {'Split': 'Test', **{k: round(test_metrics[k], 4) for k in metric_keys}},
])
print('\nMetrics summary:')
print(summary.to_string(index=False))

plot_keys = ['aspect_f1', 'sent_f1']
x      = np.arange(len(plot_keys))
width  = 0.32
fig, ax = plt.subplots(figsize=(7, 5))
for i, (split, metrics) in enumerate([('Dev', dev_metrics), ('Test', test_metrics)]):
    scores = [metrics[k] for k in plot_keys]
    bars   = ax.bar(x + (i - 0.5) * width, scores, width,
                    label=split, color=['#3498db','#e74c3c'][i], alpha=0.85)
    ax.bar_label(bars, fmt='%.3f', fontsize=9, padding=3)
ax.set_xticks(x); ax.set_xticklabels(['Aspect F1 (micro)', 'Sentiment F1 (macro)'], fontsize=10)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('Dev vs Test Metrics')
ax.legend(); ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

# ── 3. Per-category Aspect F1 (Test) ─────────────────────────────────────
per_cat_f1 = [
    sk_f1(test_metrics['asp_gold'][:, i], test_metrics['asp_pred'][:, i], zero_division=0)
    for i in range(NUM_ASPECTS)
]
short_labels = [c.replace('_', ' ') for c in ASPECT_CATEGORIES]  # underscore → space

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(short_labels, per_cat_f1, color='#3498db', alpha=0.85)
ax.bar_label(bars, fmt='%.2f', fontsize=8, padding=3)
ax.set_ylim(0, 1.15); ax.set_ylabel('F1')
ax.set_title('Aspect Detection F1 per Category (Test)')
ax.yaxis.grid(True, linestyle='--', alpha=0.5); ax.set_axisbelow(True)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout(); plt.show()

# ── 4. Sentiment confusion matrix (Test) ─────────────────────────────────
cm      = confusion_matrix(test_metrics['sen_gold'], test_metrics['sen_pred'],
                           labels=list(range(NUM_POLARITIES)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
annots  = np.array([[f'{cm[i,j]}\n({cm_norm[i,j]:.0%})' for j in range(NUM_POLARITIES)]
                    for i in range(NUM_POLARITIES)])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_norm, annot=annots, fmt='', cmap='Blues', ax=ax,
            xticklabels=POLARITIES, yticklabels=POLARITIES,
            vmin=0, vmax=1, linewidths=1.5, linecolor='white',
            annot_kws={'size': 11, 'weight': 'bold'})
for i in range(NUM_POLARITIES):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='#2ecc71', lw=2.5))
ax.set_title('Sentiment Confusion Matrix (Test)', fontsize=12, fontweight='bold')
ax.set_ylabel('Gold'); ax.set_xlabel('Predicted')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout(); plt.show()

# ── 5. Per-class sentiment report ────────────────────────────────────────
from sklearn.metrics import classification_report
print('\nPer-class Sentiment (Test):')
print(classification_report(
    test_metrics['sen_gold'], test_metrics['sen_pred'],
    target_names=POLARITIES, zero_division=0))